# V4 (test) : un professeur traduit du bambara parlé, sur le Drive du compte de secours

Seulement le quatrième tour (§9d du notebook de la phase 3), sans rien
refaire d'autre : les professeurs passent sur ta voix, puis, si l'un d'eux
traduit mieux que le modèle en service, la v4 apprend sur ses traductions et
n'entre en service que si elle fait mieux sur le test maison.

**Le Drive du compte de secours.** Tes modèles et tes enregistrements restent
dans le dossier `bambara` du compte principal ; le compte de secours y accède
par un raccourci, et tout ce que ce notebook crée (traductions, v4, environ
4 Go) prend la place du compte de secours. À faire une fois :

1. **Compte principal**, sur drive.google.com : clic droit sur le dossier
   `bambara` → Partager → l'adresse du compte de secours, rôle **Éditeur**.
2. **Compte de secours**, sur drive.google.com : « Partagés avec moi » →
   clic droit sur `bambara` → Organiser → **Ajouter un raccourci** → Mon
   Drive (le raccourci doit s'appeler `bambara`, à la racine).
3. Ici : ouvrir Colab avec **ton compte habituel**. Quand la cellule 2
   demande l'accès à Drive, choisir **le compte de secours** dans la fenêtre.

Le GPU reste celui du compte qui ouvre Colab : seul le stockage change de
compte.

*Exécution → Modifier le type d'exécution → GPU (T4)*, puis **Exécution →
Tout exécuter**. Environ 2 h 30 de GPU si un professeur est retenu (25 min
sinon) ; après une coupure, tout relancer : chaque étape faite est sautée,
la traduction reprend où elle en était.

## 1. Installation

In [ ]:
# Branche de travail : contient les dernières corrections du notebook et des
# scripts. Une fois fusionnée, la branche par défaut suffit (sans -b).
BRANCH = "claude/chat-ia-bambara-dho8oa"

import os
os.chdir("/content")
if os.path.isdir("projet/.git"):
    !git -C projet pull -q   # relance : code mis à jour, rien à recloner
else:
    !git clone -q -b $BRANCH https://github.com/LOSTHADY/IA_Project- projet
%cd /content/projet
!pip install -q transformers datasets accelerate sentencepiece jiwer sacrebleu soundfile librosa

## 2. Le Drive du compte de secours

Dans la fenêtre qui s'ouvre : **le compte de secours**. Pour changer de
compte après coup : Exécution → Déconnecter et supprimer l'environnement
d'exécution, puis tout relancer.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import shutil
from pathlib import Path
BAMBARA = "/content/drive/MyDrive/bambara"   # le raccourci vers le dossier du compte principal
CKPT, OUT, ATELIER = f"{BAMBARA}/modeles", f"{BAMBARA}/resultats-phase3", f"{BAMBARA}/atelier"
if not Path(f"{CKPT}/nllb-bm2fr/model.safetensors").exists():
    raise RuntimeError(
        "Le dossier bambara du compte principal n'est pas visible depuis ce Drive. Sur le "
        "compte principal, partager « bambara » avec le compte de secours (Éditeur) ; sur le "
        "compte de secours, « Partagés avec moi » → clic droit sur « bambara » → Organiser → "
        "Ajouter un raccourci → Mon Drive. Puis relancer cette cellule.")
print(f"Modèles trouvés. Place libre sur ce Drive : "
      f"{shutil.disk_usage('/content/drive').free / 1e9:.1f} Go")

## 3. Réglages

`PAIRES_PROFESSEUR` : phrases parlées que le professeur traduit (30 000,
environ 45 min). Pour un premier essai plus court, 5 000 (environ 8 min) :
la v4 aura alors moins de chances de faire mieux.

In [ ]:
PROFESSEUR = True
PAIRES_PROFESSEUR = 30_000

In [ ]:
# Modèle final s'il existe, sinon le dernier checkpoint (entraînement coupé
# ou inachevé), sinon le modèle d'origine quand il y en a un. Défini ici pour
# rester disponible après une reprise (sections 1 à 3) ; les sections 6 à 10
# appellent chemins().
from pathlib import Path
from transformers.trainer_utils import get_last_checkpoint

def modele(dossier, repli=None):
    if (Path(dossier) / "model.safetensors").exists():
        return dossier
    dernier = get_last_checkpoint(dossier) if Path(dossier).is_dir() else None
    if dernier:
        print(f"Entraînement inachevé : {dernier}")
        return dernier
    if repli:
        print(f"Pas encore entraîné ({dossier}) : modèle d'origine {repli}")
        return repli
    raise FileNotFoundError(f"ni modèle final ni checkpoint dans {dossier}")

def deja(label):
    """Vrai si un rapport porte déjà ce nom : relancer une cellule
    d'évaluation, d'une session à l'autre, ne la refait pas. Sauf si l'un
    de ses modèles a été réentraîné depuis : l'ancien rapport est alors mis
    de côté (.ancien) et l'évaluation refaite."""
    import glob, json, os
    for p in glob.glob(f"{OUT}/*.json"):
        try:
            rapport = json.load(open(p))
        except ValueError:  # copie vers Drive interrompue : rapport tronqué
            os.rename(p, p + ".tronque")
            continue
        if rapport.get("nom") != label:
            continue
        poids = [f for m in rapport.get("modeles", {}).values() if isinstance(m, str)
                 for f in glob.glob(f"{m}/*.safetensors")]
        if any(os.path.getmtime(f) > os.path.getmtime(p) for f in poids):
            for ancien in (p, p[:-len(".json")] + ".details.jsonl"):
                if os.path.exists(ancien):
                    os.rename(ancien, ancien + ".ancien")
            print(f"« {label} » évalué avec un modèle réentraîné depuis : refait.")
            return False
        print(f"« {label} » déjà évalué ({p}) : cellule sautée.")
        return True
    return False

def a_jour(motif, code=False):
    """Vrai si la sortie la plus récente qui correspond au motif est
    postérieure aux poids de tous les modèles en service (chemins()) et,
    avec code=True, au dernier commit qui modifie la chaîne (bambara_voice) :
    sinon, l'étape est à refaire."""
    import glob, os, subprocess
    sorties = glob.glob(motif)
    if not sorties:
        return False
    modeles = {*chemins(), *meilleurs()}
    reperes = [os.path.getmtime(f) for m in modeles for f in glob.glob(f"{m}/*.safetensors")]
    if code:
        commit = subprocess.run(["git", "log", "-1", "--format=%ct", "--", "bambara_voice"],
                                capture_output=True, text=True).stdout.strip()
        reperes += [float(commit)] if commit else []
    derniere = max(os.path.getmtime(f) for f in sorties)
    return all(r < derniere for r in reperes)

def place_drive(besoin_go):
    """Arrête la cellule si Drive n'a pas la place : un Drive plein ne
    reçoit plus rien, sans aucune erreur dans Colab."""
    import shutil
    libre = shutil.disk_usage("/content/drive").free / 1e9
    print(f"Place libre sur Drive : {libre:.1f} Go (il en faut {besoin_go})")
    if libre < besoin_go:
        raise RuntimeError("Drive presque plein. Les fichiers supprimés restent dans la "
                           "corbeille et comptent encore : la vider sur drive.google.com.")

def synchroniser_drive(minutes=30):
    """Attend que Drive ait reçu tout ce qui a été écrit, puis le remonte
    (Colab peut redemander l'accès : accepter la fenêtre qui s'ouvre).
    Ne lève jamais d'erreur : « Tout exécuter » continue."""
    try:
        drive.flush_and_unmount(timeout_ms=minutes * 60 * 1000)
        print("Drive a tout reçu.")
    except Exception as e:
        print(f"Drive n'a pas tout reçu en {minutes} min ({e}). Drive plein ? Vider la "
              "corbeille sur drive.google.com, puis relancer cette cellule.")
    try:
        drive.mount("/content/drive")
    except Exception as e:
        print(f"Drive non remonté ({e}) : relancer la section 2 avant la démo.")

def chemins():
    """Whisper, NLLB bambara → français et NLLB français → bambara à
    utiliser, d'après ce qui est sur Drive."""
    nllb = "facebook/nllb-200-distilled-600M"
    return (modele(f"{CKPT}/whisper-small-bm"),
            modele(f"{CKPT}/nllb-bm2fr", repli=nllb),
            modele(f"{CKPT}/nllb-fr2bm", repli=nllb))

RETENU = f"{CKPT}/retenu.json"           # l'assemblage en service et ses preuves
MOYENNE = "/content/nllb-bm2fr-moyenne"  # moyenne des poids v1/v2 (§9a), refaite au besoin

def en_service():
    """L'assemblage en service (eval.promotion) : Whisper + NLLB v1, celui de
    la démo du 15 octobre, tant qu'aucun candidat n'a fait mieux, mesure à
    l'appui, sur le test maison. Un nouveau modèle ne le remplace que là."""
    from eval.promotion import lire_retenu
    whisper, bm2fr, fr2bm = chemins()
    r = lire_retenu(RETENU, {"asr": {"model_id": whisper}, "nom_asr": "Whisper",
                             "bm2fr": bm2fr, "nom_bm2fr": "NLLB v1",
                             "fr2bm": fr2bm, "nom_fr2bm": "NLLB v1"})
    if r["bm2fr"] == MOYENNE and not Path(f"{MOYENNE}/model.safetensors").exists():
        import subprocess  # la moyenne vit sur le disque de Colab : une minute
        subprocess.run(["python", "scripts/moyenne_poids.py", "--a", bm2fr,
                        "--b", f"{CKPT}/nllb-bm2fr-v2", "--out", MOYENNE], check=True)
    return r

def meilleurs():
    """Les modèles de l'assistant (§9 à 11) : Whisper (v2 si entraîné) et les
    deux NLLB en service."""
    r = en_service()
    whisper = chemins()[0]
    if Path(f"{CKPT}/whisper-small-bm-v2/model.safetensors").exists():
        whisper = f"{CKPT}/whisper-small-bm-v2"
    return whisper, r["bm2fr"], r["fr2bm"]

## 4. Ce qui est en service, mesuré sur ta voix

L'assemblage de `retenu.json` (Soloni + NLLB v1 depuis le 17 octobre) et sa
mesure sur le jeu maison (§9a de la phase 3), reprise telle quelle ; refaite
seulement si tu as enregistré de nouvelles phrases depuis (une dizaine de
minutes).

In [ ]:
import glob, json, os
from eval.promotion import score
WHISPER, NLLB_BM2FR, NLLB_FR2BM = meilleurs()
EN_SERVICE = en_service()
CFG = "/content/assistant-affine.json"
import torch  # sans GPU, sur processeur : plus lent, mais sans quota
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
json.dump({"device": DEVICE, "asr": EN_SERVICE["asr"], "mt_in": {"model_id": NLLB_BM2FR},
           "mt_out": {"model_id": NLLB_FR2BM}}, open(CFG, "w"), indent=1)
MAISON, PRISES = f"{ATELIER}/testset-dire.jsonl", f"{ATELIER}/dire.jsonl"
TITULAIRE = f"{EN_SERVICE['nom_asr']} + {EN_SERVICE['nom_bm2fr']}"
RAPPORT_EN_SERVICE = None
if not os.path.exists(PRISES):
    print("Aucune phrase enregistrée avec l'atelier : la v4 ne peut pas être jugée.")
else:
    if EN_SERVICE["asr"].get("kind") == "nemo":   # Soloni : jugement de la v4 au §5
        !pip install -q "nemo_toolkit[asr]"
    !python -m eval.atelier --racine "$ATELIER" exporter > /dev/null
    dossier = f"{OUT}/maison-assemblages/{EN_SERVICE['nom_asr']}-{EN_SERVICE['nom_bm2fr']}".replace(" ", "-")
    motif = f"{dossier}/cascade-*[0-9].json"
    if not (glob.glob(motif) and max(map(os.path.getmtime, glob.glob(motif))) > os.path.getmtime(PRISES)):
        print(f"{TITULAIRE} : mesure sur tes phrases…")
        !python -m eval.run_eval --testset "$MAISON" --arch cascade --config $CFG \
            --comprendre-seulement --nom "$TITULAIRE" --out "$dossier" > /dev/null 2>&1
    if glob.glob(motif):
        RAPPORT_EN_SERVICE = sorted(glob.glob(motif))[-1]
        print(f"En service : {TITULAIRE} ; chrF++ sur ta voix : dev "
              f"{score(RAPPORT_EN_SERVICE, 'dev'):.1f}, test {score(RAPPORT_EN_SERVICE, 'test'):.1f}")
    else:
        print(f"⚠️ {TITULAIRE} n'a pas pu être mesuré : relancer la cellule.")

## 5. Quatrième tour

### Juger un nouveau traducteur bambara → français

Toujours de la même façon, pour la v3 (§9c) comme pour la v4 (§9d) :

1. le nouveau traducteur traduit ce que la reconnaissance en service a
   entendu de tes phrases (jeu maison) ;
2. s'il ne fait pas mieux que le modèle en service **sur le dev**, il est
   refusé tout de suite ;
3. sinon, contrôles écrits (Bayelemabaga, MAFAND) : refusé s'il y perd plus
   d'un point de façon établie ;
4. il entre en service seulement s'il fait mieux **sur le test**, IC à 95 %
   au-dessus de zéro. La décision va dans `retenu.json`.

In [ ]:
import glob, json, os, shutil, subprocess, sys
from pathlib import Path
from eval.atelier import trouver_rapport
from eval.promotion import comparer, controle, ecrire_retenu, score, to_markdown, verdict

DEV_MAISON = "/content/maison-dev.jsonl"
MAFAND = "/content/donnees-v3/mafand"

def lancer(*args):
    """python ARGS ; la sortie n'est montrée qu'en cas d'échec."""
    r = subprocess.run([sys.executable, *args], capture_output=True, text=True)
    if r.returncode:
        print(f"⚠️ Échec : python {' '.join(args)}\n{(r.stdout + r.stderr)[-2000:]}")
    return r.returncode == 0

def dev_maison():
    """Le dev maison tel que la reconnaissance en service l'a transcrit : le
    checkpoint d'un nouveau traducteur est choisi dessus (--eval-file), sur
    la parole quotidienne et non sur des contes ou des griots."""
    lignes = [json.loads(l) for l in open(RAPPORT_EN_SERVICE.replace(".json", ".details.jsonl"))]
    dev = [{"bm": r["source_bm"], "fr": r["ref_translation_fr"]} for r in lignes
           if r.get("partition") == "dev" and r.get("source_bm")]
    with open(DEV_MAISON, "w") as f:
        f.write("\n".join(json.dumps(r, ensure_ascii=False) for r in dev))
    print(f"Dev maison : {len(dev)} énoncés transcrits par {EN_SERVICE['nom_asr']}")

def vers_drive(local, dossier):
    """Le modèle final, sans ses checkpoints (trop lourds pour Drive)."""
    os.makedirs(dossier, exist_ok=True)
    for f in Path(local).iterdir():
        if f.is_file():
            shutil.copy2(f, dossier)
    print(f"Modèle enregistré dans {dossier}")

def juger_bm2fr(chemin, version):
    """Le traducteur `chemin` remplace celui en service seulement s'il fait
    mieux (voir ci-dessus) ; renvoie la décision, None si une mesure manque."""
    global EN_SERVICE, RAPPORT_EN_SERVICE, WHISPER, NLLB_BM2FR, NLLB_FR2BM
    asr = EN_SERVICE["nom_asr"]
    titulaire, nom = f"{asr} + {EN_SERVICE['nom_bm2fr']}", f"{asr} + NLLB {version}"
    candidat, maillon = {"bm2fr": chemin, "nom_bm2fr": f"NLLB {version}"}, f"bambara → français ({version})"
    # 1. Sur le jeu maison, avec la même reconnaissance.
    dossier = f"{OUT}/maison-assemblages/{asr}-NLLB-{version}".replace(" ", "-")
    cfg = f"/content/assemblage-{version}.json"
    json.dump({"device": "cuda", "asr": EN_SERVICE["asr"], "mt_in": {"model_id": chemin},
               "mt_out": {"model_id": EN_SERVICE["fr2bm"]}}, open(cfg, "w"), indent=1)
    motif = f"{dossier}/cascade-*[0-9].json"
    if not (a_jour(motif, code=True) and max(map(os.path.getmtime, glob.glob(motif))) > os.path.getmtime(PRISES)):
        print(f"{nom} : mesure sur tes phrases…")
        lancer("-m", "eval.run_eval", "--testset", MAISON, "--arch", "cascade", "--config", cfg,
               "--comprendre-seulement", "--nom", nom, "--out", dossier)
    if not glob.glob(motif):
        print(f"⚠️ {nom} : pas de rapport, la mesure a échoué (relancer la cellule).")
        return None
    rapport = sorted(glob.glob(motif))[-1]
    # 2. Pas même meilleur sur le dev : ce n'est pas un candidat (règle du
    # §9a), inutile de dépenser les contrôles écrits.
    dev, dev_titulaire = score(rapport, "dev"), score(RAPPORT_EN_SERVICE, "dev")
    if dev <= dev_titulaire:
        decision = {"promu": False, "dev": {version: dev, "en service": dev_titulaire},
                    "raisons": [f"pas meilleure sur le dev maison : {dev:.1f} contre "
                                f"{dev_titulaire:.1f} pour {titulaire}"]}
        print(f"**Refusé : {titulaire} reste en service.** " + decision["raisons"][0])
        EN_SERVICE = ecrire_retenu(RETENU, EN_SERVICE, maillon, candidat, decision)
        return decision
    # 3. Contrôles écrits : le modèle en service et le candidat, mêmes phrases.
    if not Path(f"{MAFAND}/test.jsonl").exists():
        lancer("scripts/donnees_v3.py", "--out", str(Path(MAFAND).parent))
    controles = []
    for corpus, jeu in (("Bayelemabaga", "RobotsMaliAI/bayelemabaga"), ("MAFAND", MAFAND)):
        rapports = {}
        for qui, b2f in (("en service", EN_SERVICE["bm2fr"]), (version, chemin)):
            label = f"controle {corpus} {Path(b2f).name}"
            if trouver_rapport(f"{OUT}/controles", label) is None:
                print(f"Contrôle {corpus} : {Path(b2f).name}…")
                lancer("-m", "eval.baselines", "--out", f"{OUT}/controles", "--label", label, "mt",
                       "--dataset", jeu, "--limit", "500", "--bm2fr-model", b2f,
                       "--fr2bm-model", NLLB_FR2BM)
            rapports[qui] = trouver_rapport(f"{OUT}/controles", label)
        if None in rapports.values():
            print(f"⚠️ Contrôle {corpus} manquant : relancer la cellule.")
            return None
        controles.append(controle(rapports[version], rapports["en service"], "chrF++ bm→fr"))
    # 4. Verdict sur le test : en service seulement s'il fait mieux.
    decision = verdict(comparer(rapport, RAPPORT_EN_SERVICE, "test"), controles)
    print(to_markdown(decision, nom, titulaire))
    EN_SERVICE = ecrire_retenu(RETENU, EN_SERVICE, maillon, candidat, decision)
    if decision["promu"]:
        RAPPORT_EN_SERVICE = rapport
    WHISPER, NLLB_BM2FR, NLLB_FR2BM = meilleurs()
    json.dump({"device": "cuda", "asr": EN_SERVICE["asr"], "mt_in": {"model_id": EN_SERVICE["bm2fr"]},
               "mt_out": {"model_id": EN_SERVICE["fr2bm"]}}, open(CFG, "w"), indent=1)
    return decision

### 9d. Quatrième tour : un professeur traduit du bambara parlé, la v4 apprend

La v3 n'a rien gagné : ses données (presse, dictionnaire) ne ressemblent pas
à ta façon de parler. Ce qui manque, ce sont des phrases de tous les jours
avec leur français. RobotsMali publie les premières sans le second : les
transcriptions relues d'**AfVoices** (conversations, CC-BY-4.0) et de
**Kunkado** (radio, CC-BY-SA-4.0). Un **professeur**, traducteur ouvert plus
gros que le nôtre, leur donne un français ; la v4 apprend sur ces paires
(distillation : Meta a obtenu notre NLLB 600M ainsi, à partir de son
modèle de 54 milliards de paramètres).

1. **Les professeurs passent d'abord sur ta voix** : chacun traduit ce que
   la reconnaissance en service a entendu de tes phrases, comme le modèle en
   service lui-même (témoin, même script). Le meilleur sur le dev n'est
   retenu que s'il bat le modèle en service de façon établie : sinon ses
   traductions n'apprendraient rien, et on s'arrête là (environ 25 min,
   téléchargements compris).
2. **Traduction** de `PAIRES_PROFESSEUR` phrases parlées (environ 45 min
   pour 30 000 ; reprise automatique après une coupure, tout est sur Drive).
3. **Filtrage** des traductions douteuses (vides, en boucle, non traduites,
   de longueur aberrante), puis **entraînement** de la v4 à partir du modèle
   en service, avec un rappel de Bayelemabaga (environ 1 h). La source de
   chaque paire est écrite comme Soloni écrit (sans ponctuation) : la v4
   apprend sur ce qu'elle recevra au micro.
4. **Jugement** comme pour la v3 (ci-dessus).

Seule la source des données change par rapport au modèle en service : si la
v4 fait mieux, c'est grâce à elles. Les paires restent sur Drive (licences
non commerciale de NLLB et à partage identique de Kunkado), jamais dans le
dépôt.

In [ ]:
import glob, json, os, shutil
from datetime import datetime
from pathlib import Path
from eval.promotion import choisir, comparer, score
PROFESSEURS = {"NLLB 3.3B": "facebook/nllb-200-3.3B",
               "NLLB 1.3B distillé": "facebook/nllb-200-distilled-1.3B",
               "MADLAD 3B": "google/madlad400-3b-mt"}
PROF_DIR = f"{CKPT}/../professeur"   # corpus et traductions, sur Drive
PROF = None                          # (nom, modèle) du professeur retenu
if not PROFESSEUR:
    print("PROFESSEUR = False (§3) : section sautée.")
elif not RAPPORT_EN_SERVICE:
    print("Pas de mesure du jeu maison (§9a) : section sautée.")
else:
    asr, TEMOIN = EN_SERVICE["nom_asr"], f"{EN_SERVICE['nom_bm2fr']} (témoin, même script)"
    rapports_prof = {}
    for nom, modele_id in {**PROFESSEURS, TEMOIN: EN_SERVICE["bm2fr"]}.items():
        dossier = f"{OUT}/professeurs/{asr}-{nom}".replace(" ", "-")
        for signe in "(),":
            dossier = dossier.replace(signe, "")
        motif = f"{dossier}/cascade-*[0-9].json"
        # À refaire quand la reconnaissance a réentendu tes phrases (§9a).
        if not (glob.glob(motif) and max(map(os.path.getmtime, glob.glob(motif)))
                > os.path.getmtime(RAPPORT_EN_SERVICE)):
            print(f"{nom} : traduction de tes phrases…")
            !python scripts/professeurs.py mesurer --rapport "$RAPPORT_EN_SERVICE" \
                --modele "$modele_id" --nom "$nom" --out "$dossier"
        if glob.glob(motif):
            rapports_prof[nom] = sorted(glob.glob(motif))[-1]
        else:
            print(f"⚠️ {nom} : pas de rapport (relancer la cellule).")

    print(f"\n| Traduction de ce que {asr} a entendu | chrF++ dev |\n|---|---|")
    print(f"| {EN_SERVICE['nom_bm2fr']} (en service, §9a) | {score(RAPPORT_EN_SERVICE, 'dev'):.1f} |")
    for nom, r in sorted(rapports_prof.items(), key=lambda m: -score(m[1], "dev")):
        print(f"| {nom} | {score(r, 'dev'):.1f} |")
    # Les professeurs sont comparés au témoin : même script, mêmes réglages.
    reference = rapports_prof.get(TEMOIN, RAPPORT_EN_SERVICE)
    if abs(score(reference, "dev") - score(RAPPORT_EN_SERVICE, "dev")) > 1:
        print("⚠️ Le témoin s'écarte de plus d'un point de la mesure du §9a : à signaler.")
    profs = {n: r for n, r in rapports_prof.items() if n in PROFESSEURS}
    choix = {"date": datetime.now().isoformat(timespec="seconds"), "reconnaissance": asr,
             "dev": {n: score(r, "dev") for n, r in rapports_prof.items()}, "retenu": None}
    if profs:
        meilleur = choisir(profs, "dev")
        c = comparer(profs[meilleur], reference, "dev")
        ecart = f"{c['ecart']:+.1f} [{c['ic'][0]:.1f} ; {c['ic'][1]:.1f}]"
        choix["comparaison"] = {"professeur": meilleur, **c}
        if c["meilleur"]:
            PROF = (meilleur, PROFESSEURS[meilleur])
            choix["retenu"] = meilleur
            print(f"\nProfesseur retenu : {meilleur}, meilleur que {EN_SERVICE['nom_bm2fr']} "
                  f"sur le dev : {ecart} chrF++.")
        else:
            print(f"\nAucun professeur ne traduit ta voix mieux que {EN_SERVICE['nom_bm2fr']} "
                  f"de façon établie (le meilleur, {meilleur} : {ecart}). Ses traductions "
                  "n'apprendraient rien à notre modèle : on s'arrête là, sans entraînement.")
    os.makedirs(f"{OUT}/professeurs", exist_ok=True)
    json.dump(choix, open(f"{OUT}/professeurs/choix.json", "w"), ensure_ascii=False, indent=1)
    # Les professeurs écartés rendent leur place (jusqu'à 18 Go) au disque de Colab.
    for modele_id in PROFESSEURS.values():
        if PROF is None or modele_id != PROF[1]:
            shutil.rmtree(Path.home() / ".cache" / "huggingface" / "hub"
                          / ("models--" + modele_id.replace("/", "--")), ignore_errors=True)
CORPUS = f"{PROF_DIR}/bm.jsonl"
PAIRES_PROF = f"{PROF_DIR}/paires-{PROF[0]}.jsonl".replace(" ", "-") if PROF else None

In [ ]:
from pathlib import Path
if PROF is None:
    print("Pas de professeur retenu : rien à traduire.")
else:
    # Une fois pour toutes, sur Drive : après une coupure, la reprise
    # retrouve les mêmes phrases dans le même ordre.
    if not Path(CORPUS).exists():
        !python scripts/donnees_robotsmali.py corpus --out "$PROF_DIR"
    PROF_MODELE = PROF[1]
    !python scripts/professeurs.py traduire --entree "$CORPUS" --modele "$PROF_MODELE" \
        --out "$PAIRES_PROF" --max $PAIRES_PROFESSEUR

In [ ]:
import os
from pathlib import Path
B2F_V4, LOCAL_V4, DONNEES_V4 = f"{CKPT}/nllb-bm2fr-v4", "/content/nllb-bm2fr-v4", "/content/donnees-v4"
if PROF is None:
    print("Pas de professeur retenu : pas de v4.")
else:
    voulues = min(PAIRES_PROFESSEUR, sum(1 for _ in open(CORPUS))) if os.path.exists(CORPUS) else 1
    faites = sum(1 for _ in open(PAIRES_PROF)) if os.path.exists(PAIRES_PROF) else 0
    if faites < voulues:
        print(f"Traduction inachevée ({faites}/{voulues}) : relancer la cellule précédente, "
              "elle reprend où elle en était.")
    else:
        # 3. Filtrage, puis entraînement à partir du modèle en service.
        !python scripts/donnees_robotsmali.py filtrer --entree "$PAIRES_PROF" --out "$DONNEES_V4/robotsmali"
        dev_maison()
        DEPART = EN_SERVICE["bm2fr"]
        if not Path(f"{B2F_V4}/model.safetensors").exists():
            place_drive(3)
            !python scripts/finetune_nllb.py --direction bm2fr --model "$DEPART" --main-samples 10000 \
                --extra-dataset "$DONNEES_V4/robotsmali" --eval-file "$DEV_MAISON" \
                --epochs 1 --save-steps 200 --output "$LOCAL_V4" --resume
            if Path(f"{LOCAL_V4}/model.safetensors").exists():
                vers_drive(LOCAL_V4, B2F_V4)
        # 4. Jugée comme la v3.
        if Path(f"{B2F_V4}/model.safetensors").exists() and EN_SERVICE["bm2fr"] != B2F_V4:
            juger_bm2fr(B2F_V4, "v4")
    print(f"En service : {EN_SERVICE['nom_asr']} + {EN_SERVICE['nom_bm2fr']}")

## Ensuite

- **Me montrer** le tableau des professeurs et la décision sur la v4.
- **La démo** (`notebooks/assistant.ipynb`) lit `retenu.json` : si la v4 est
  en service, elle l'utilise. Elle monte aussi Drive : y choisir le même
  compte de secours (ou le principal, qui voit aussi la v4 dans son dossier).